# Forge: fine-tune Qwen3 into a local coding/shell assistant

**Settings (right panel):** Accelerator **GPU T4 x2** (or P100), Internet **On**, Persistence: Files.

Pick the model size in the next cell. Rough times on one T4 for the default data mix:
`qwen3-8b` about 3-5 h, `qwen3-14b` about 7-10 h (the script saves and stops before the 12 h limit).

In [ ]:
PRESET = "qwen3-8b"      # or "qwen3-14b"
QUANT = "q4_k_m"         # q4_k_m (balanced) | iq4_xs / q3_k_m (smaller, faster on a 4 GB GPU)
REPO = "https://github.com/manian1909/local-assistant.git"
OUT = f"/kaggle/working/forge-{PRESET}"


In [ ]:
%%capture
!pip install -q unsloth
!pip install -q ddgs httpx

In [ ]:
!git clone -q $REPO /kaggle/working/local-assistant
%cd /kaggle/working/local-assistant
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build the training mix
Public chat/code/function-calling data is downloaded here; `data/out/agentic.jsonl` (our own tool-use conversations) comes from the repo.

In [ ]:
!python data/build_dataset.py

## 2. Fine-tune (QLoRA)
To resume after a session ends: attach the previous notebook output as input and add `--resume-from /kaggle/input/<name>/forge-.../checkpoints/checkpoint-XXX`.

In [ ]:
!python training/finetune.py --preset $PRESET --out $OUT --max-hours 10.5

## 3. Export to GGUF
Merges the adapter into the base model and quantizes it. Intermediate files go to `/kaggle/tmp` because `/kaggle/working` is limited to 20 GB.

In [ ]:
!mkdir -p /kaggle/tmp
!python training/export.py $OUT/lora --quant $QUANT --out /kaggle/tmp/gguf --copy-to /kaggle/working

If the merged export fails (out of disk or RAM, mostly with 14B), export just the adapter instead and combine it with the official base GGUF locally:

In [ ]:
# !python training/export.py $OUT/lora --method lora --out /kaggle/working

## 4. Clean up and download
Delete training checkpoints so the output fits, then download the `.gguf` file from the **Output** panel.

In [ ]:
!rm -rf $OUT/checkpoints /kaggle/working/local-assistant
!ls -lh /kaggle/working /kaggle/working/*/ 2>/dev/null